<a href="https://www.kaggle.com/code/mukeshkumar95/data-augmentation-using-smote?scriptVersionId=369403329" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [2]:
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix

from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline

# Creating imbalanced data

In [3]:
X,y =make_classification(n_samples=1000,
                        n_features=2,
                        n_informative=2,
                        n_redundant=0,
                        weights=[0.99,0.01],
                        random_state=42)



In [4]:
print(X)

[[-0.05861915 -0.35109008]
 [ 0.94927735 -1.22261256]
 [ 0.59591118 -1.2900237 ]
 ...
 [-0.10126746 -1.59017563]
 [ 0.64376105 -0.78053855]
 [ 0.69852984 -1.27940455]]


In [5]:
X_train,X_test,y_train, y_test=train_test_split(X,y, test_size=0.3, random_state=42, stratify=y)

# Baseline Model (no fix)

In [6]:
baseline=LogisticRegression()
baseline.fit(X_train,y_train)

pred_base=baseline.predict(X_test)
print("Baseline (no fix)")
print(confusion_matrix(y_test, pred_base))
print(classification_report(y_test, pred_base))

Baseline (no fix)
[[295   0]
 [  5   0]]
              precision    recall  f1-score   support

           0       0.98      1.00      0.99       295
           1       0.00      0.00      0.00         5

    accuracy                           0.98       300
   macro avg       0.49      0.50      0.50       300
weighted avg       0.97      0.98      0.98       300



/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


# Adding SMOTE using a pipeline

In [7]:
smote_clf=Pipeline(steps=[
    ("smote",SMOTE(random_state=42, k_neighbors=3)),
    ("model", LogisticRegression())
])

In [8]:
smote_clf.fit(X_train, y_train)
pred_smote=smote_clf.predict(X_test)

print("SMOTE + Logistic Regression")
print(confusion_matrix(y_test, pred_smote))
print(classification_report(y_test, pred_smote))

SMOTE + Logistic Regression
[[229  66]
 [  1   4]]
              precision    recall  f1-score   support

           0       1.00      0.78      0.87       295
           1       0.06      0.80      0.11         5

    accuracy                           0.78       300
   macro avg       0.53      0.79      0.49       300
weighted avg       0.98      0.78      0.86       300

